# SmolVLA fine-tuned on dataset v1 (200 demonstrations), Kaggle

Fine-tunes `lerobot/smolvla_base` on `Kimaa6/icub_gardener_v1` with **the same data as ACT v1**: the 180 training
and 20 validation episodes of `configs/act_split.json`, training seed 1000.
- **Inputs:** the 3 cameras, the 12 joint values and the instruction as a sentence, for example
  "Give the left plant just a little water.".
- **No task-ID vector**, unlike ACT: the pot and the amount must come from the words.
- **Settings:** LeRobot's SmolVLA defaults. The vision encoder and the language model are frozen; only the action
  expert learns. lr 1e-4 with cosine decay, chunk of 50 actions = 2 s.
- **Batch and steps** are chosen by cell 6 from a measurement on this GPU, so the run fits in one session.

**Before running:**
1. Notebook settings: **Accelerator GPU T4 x1** (not P100: no bfloat16), **Internet on**, the secret **`HF_TOKEN`**
   (read access to the dataset), and **Persistence: Files** off.
2. **First, measure:** in an interactive session, run cells 1-6 only (about 25 min, most of it downloads).
   Cell 6 prints the peak GPU memory, the seconds per step and the plan. Check them before the long run.
3. **Then the long run:** **Save Version, Save & Run All (Commit)**. It runs on its own with the browser closed,
   up to 12 h. Cell 6 measures again and picks the same plan.

**Output** (Output tab):
- `smolvla_<step>.zip`: one zip for each of the 3 checkpoints with the lowest validation loss, about 0.85 GB each.
  Download them one at a time.
- `smolvla_logs.zip`: the training log, the measurement and the validation table.

In [ ]:
# 1. GPU and versions
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python --version

In [ ]:
# 2. Code: the public repo, main branch. The commit is printed so the run can be traced.
!rm -rf /kaggle/working/icub-gardener
!git clone -q -b main https://github.com/kaarima/icub-gardener.git /kaggle/working/icub-gardener
!cd /kaggle/working/icub-gardener && git log --oneline -1

In [ ]:
# 3. LeRobot 0.4.4 with its SmolVLA extra. The extra pins transformers to 4.57-4.x: Kaggle's transformers 5.x does not
# work with LeRobot 0.4.4 (that broke the first ACT v2 run), so the pip line replaces it.
# '!' lines do not stop the run when they fail, so the import test runs with check=True: a failure stops here.
import subprocess
!pip install -q "lerobot[smolvla]==0.4.4"
subprocess.run(["python", "-c", "import lerobot, torch, transformers, lerobot.configs.train, lerobot.policies.smolvla.modeling_smolvla; "
                "print('lerobot', lerobot.__version__, '| torch', torch.__version__, '| transformers', transformers.__version__, "
                "'| cuda', torch.cuda.is_available(), '| bf16', torch.cuda.is_bf16_supported()); "
                "assert transformers.__version__.startswith('4.'), transformers.__version__"], check=True)

In [ ]:
# 4. Dataset v1 from the Hub (private: needs the HF_TOKEN secret)
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
!hf download Kimaa6/icub_gardener_v1 --repo-type dataset --local-dir /kaggle/working/gardener_v1 --quiet
!du -sh /kaggle/working/gardener_v1

In [ ]:
# 5. Checks before spending GPU hours: 200 episodes, ACT v1's split, the 6 instructions as sentences, one frame decodes.
import json
from lerobot.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata

REPO, ROOT = "Kimaa6/icub_gardener_v1", "/kaggle/working/gardener_v1"
split = json.load(open("/kaggle/working/icub-gardener/configs/act_split.json"))
meta = LeRobotDatasetMetadata(REPO, root=ROOT)
train, val = set(split["train"]), set(split["val"])
assert meta.total_episodes == 200, meta.total_episodes
assert len(train) == 180 and len(val) == 20 and not train & val and train | val == set(range(200)), (len(train), len(val))
tasks = sorted(meta.tasks.index)
assert tasks == sorted(["Water the left plant.", "Water the right plant.", "Give the left plant just a little water.",
                        "Give the right plant just a little water.", "Give the left plant a lot of water.",
                        "Give the right plant a lot of water."]), tasks
item = LeRobotDataset(REPO, root=ROOT, episodes=[split["val"][0]])[0]
print({k: tuple(v.shape) for k, v in item.items() if hasattr(v, "shape")}, repr(item["task"]))
print(f"OK: {meta.total_episodes} episodes, {meta.total_frames} frames; 180 train / 20 val, none in both; tasks: {tasks}")

In [ ]:
# 6. Measure on this GPU, then plan. A short run (60 steps) at batch 32, then at 16 if 32 runs out of memory:
# peak GPU memory (nvidia-smi every 0.5 s), seconds per step (LeRobot's updt_s + data_s, averaged after warm-up).
# Plan: the batch that fits, and as many steps as fit in 9 h of training (12 h session, minus downloads, validation,
# packing and a margin), at most 20,000 (LeRobot's SmolVLA example is 20,000 steps at batch 64), in thousands.
# A checkpoint every fifth of the run: 5 checkpoints.
import json, os, re, shutil, subprocess, threading, time
REPO_DIR, W = "/kaggle/working/icub-gardener", "/kaggle/working"
base = dict(os.environ, DATASET=f"{W}/gardener_v1", REPO_ID="Kimaa6/icub_gardener_v1", SPLIT="configs/act_split.json",
            WORKERS="4", LOG="10")

def measure(batch):
    out = f"{W}/smolvla_measure_{batch}"
    shutil.rmtree(out, ignore_errors=True)
    peak, done = [0], [False]
    def poll():
        while not done[0]:
            r = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                               capture_output=True, text=True)
            peak[0] = max(peak[0], int(r.stdout.split()[0]))
            time.sleep(0.5)
    t = threading.Thread(target=poll); t.start()
    env = dict(base, OUT=out, STEPS="60", BATCH=str(batch), SAVE="100000")
    p = subprocess.run(["bash", "scripts/train_smolvla.sh"], cwd=REPO_DIR, env=env, capture_output=True, text=True)
    done[0] = True; t.join()
    log = p.stdout + p.stderr
    open(f"{W}/smolvla_measure_{batch}.log", "w").write(log)
    shutil.rmtree(out, ignore_errors=True)
    if p.returncode != 0:
        oom = "out of memory" in log.lower()
        print(f"batch {batch}: {'OUT OF MEMORY' if oom else 'FAILED'}, peak {peak[0]} MiB")
        if not oom:
            print(log[-3000:])
            raise SystemExit(f"measurement at batch {batch} failed for another reason than memory: see above")
        return None
    s = [float(a) + float(b) for a, b in re.findall(r"updt_s:([0-9.]+) data_s:([0-9.]+)", log)][2:]
    params = [l for l in log.splitlines() if "params" in l.lower()][:2]
    sec = sum(s) / len(s)
    print(f"batch {batch}: peak GPU memory {peak[0]} MiB, {sec:.2f} s per step ({1 / sec:.2f} steps/s); {params}")
    return dict(batch=batch, peak_mib=peak[0], sec_per_step=sec)

m = measure(32) or measure(16)
assert m, "neither batch 32 nor 16 fits in GPU memory"
steps = min(20000, int(9 * 3600 / m["sec_per_step"] / 1000) * 1000)
assert steps >= 5000, f"only {steps} steps fit in 9 h: too slow for a useful run, stop and rethink"
plan = dict(m, steps=steps, save=steps // 5, hours=round(steps * m["sec_per_step"] / 3600, 1),
            samples=steps * m["batch"], passes=round(steps * m["batch"] / sum(meta.episodes["length"][e] for e in train), 2))
json.dump(plan, open(f"{W}/smolvla_plan.json", "w"))
print("PLAN:", plan)
print(f"for comparison, ACT v1: 80,000 steps x batch 8 = 640,000 samples")

In [ ]:
# 7. Train with the plan. The full log goes to a file; here, the first lines, every 10th step line, every checkpoint
# line and every error line. Stops the run if training fails or a checkpoint is missing.
import glob, json, os, subprocess
plan = json.load(open("/kaggle/working/smolvla_plan.json"))
env = dict(base, OUT="/kaggle/working/smolvla", STEPS=str(plan["steps"]), BATCH=str(plan["batch"]),
           SAVE=str(plan["save"]), LOG="100")
n = 0
with open("/kaggle/working/smolvla_train.log", "w") as log:
    p = subprocess.Popen(["bash", "scripts/train_smolvla.sh"], cwd=REPO_DIR, env=env,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        log.write(line)
        n += "step:" in line
        if (line.startswith("SmolVLA:") or "Checkpoint" in line or "rror" in line or "Traceback" in line
                or "params" in line.lower() or ("step:" in line and n % 10 == 1)):
            print(line, end="", flush=True)
assert p.wait() == 0, f"training failed (exit {p.returncode}): see the end of /kaggle/working/smolvla_train.log"
ckpts = sorted(glob.glob("/kaggle/working/smolvla/checkpoints/0*"))
assert len(ckpts) == 5, f"expected 5 checkpoints, found {ckpts}"
print("training done:", [os.path.basename(c) for c in ckpts])

If a session stops before the end, start a new one, run cells 1-5, then resume instead of cells 6-7 (after attaching
the earlier output as an input and copying `smolvla` back to `/kaggle/working/`):

`!lerobot-train --config_path=/kaggle/working/smolvla/checkpoints/last/pretrained_model/train_config.json --resume=true`

In [ ]:
# 8. Validation and training loss for every checkpoint, on ACT v1's split. For SmolVLA this is its own flow-matching
# loss with fixed noise: it ranks its checkpoints, it is not comparable with ACT's numbers.
!cd /kaggle/working/icub-gardener && python scripts/val_loss.py /kaggle/working/smolvla/checkpoints/0*/pretrained_model \
   --root /kaggle/working/gardener_v1 --repo-id Kimaa6/icub_gardener_v1 --split configs/act_split.json \
   2>&1 | grep -v Warn | tee /kaggle/working/smolvla_val.txt

In [ ]:
# 9. Package the 3 checkpoints with the lowest validation loss, one zip each (the laptop has little free disk),
# without the optimizer state: only pretrained_model is needed to evaluate.
import re, subprocess
rows = re.findall(r"checkpoints/(\d+)/pretrained_model: validation ([0-9.]+)", open("/kaggle/working/smolvla_val.txt").read())
assert len(rows) == 5, f"expected a validation loss for 5 checkpoints, got {rows}: see smolvla_val.txt"
best = sorted(rows, key=lambda r: float(r[1]))[:3]
print("lowest validation loss:", best)
for s, _ in best:
    subprocess.run(["zip", "-qr0", f"smolvla_{s}.zip", f"smolvla/checkpoints/{s}/pretrained_model"], cwd="/kaggle/working", check=True)
subprocess.run("zip -q smolvla_logs.zip smolvla_train.log smolvla_val.txt smolvla_plan.json smolvla_measure_*.log",
               shell=True, cwd="/kaggle/working", check=True)
!ls -lh /kaggle/working/*.zip